# AIOps Incident Embedding Pipeline

The **understanding layer** (showroom lab pillar: *inference*). This notebook:
1. pulls firing/recent **Prometheus alerts** (Thanos) and **pod events**,
2. **embeds** them into pgvector (the `aiops` logical DB in the base stack's Aurora),
3. **retrieves similar past incidents** for any new event,
4. drafts a **natural-language incident summary** via the self-hosted llama-3-1-8b — through **Portkey** (lesson L4: never direct to vLLM),
5. exports precedent facts as an **AAP inventory source**, so Lightspeed generates playbooks with real context.

Config arrives via the `aiops-vector-db` Secret and `aiops-notebook-env` ConfigMap — nothing sensitive lives in this file.

In [ ]:
%pip install -q psycopg2-binary sentence-transformers kubernetes requests

In [ ]:
import os, json, datetime, requests, psycopg2

PG = dict(host=os.environ['PGHOST'], port=os.environ.get('PGPORT', '5432'),
          dbname=os.environ['PGDATABASE'], user=os.environ['PGUSER'],
          password=os.environ['PGPASSWORD'])
PORTKEY = os.environ['PORTKEY_ENDPOINT']          # ALL LLM calls go through this (L4)
MODEL = os.environ.get('LLM_MODEL', 'llama-3-1-8b')
THANOS = os.environ.get('THANOS_URL', 'https://thanos-querier.openshift-monitoring.svc:9091')
SA_TOKEN = open('/var/run/secrets/kubernetes.io/serviceaccount/token').read()
CA = '/var/run/secrets/kubernetes.io/serviceaccount/ca.crt'
print('config loaded — db:', PG['dbname'], '| gateway:', PORTKEY)

In [ ]:
# Schema — pgvector extension is created by the aurora-db bootstrap job.
DDL = '''
CREATE TABLE IF NOT EXISTS incidents (
  id          BIGSERIAL PRIMARY KEY,
  occurred_at TIMESTAMPTZ NOT NULL DEFAULT now(),
  source      TEXT NOT NULL,          -- alertmanager | pod-event | eda | aap
  kind        TEXT NOT NULL,          -- e.g. KubePodCrashLooping, httpd-config
  namespace   TEXT,
  subject     TEXT,                   -- pod/node/pvc/app name
  raw         JSONB NOT NULL,
  rca         TEXT,                   -- AI root-cause analysis, if any
  remediation TEXT,                   -- playbook that ran / was generated
  outcome     TEXT,                   -- success | failure | escalated
  embedding   vector(384)             -- all-MiniLM-L6-v2
);
CREATE INDEX IF NOT EXISTS incidents_embedding_idx
  ON incidents USING hnsw (embedding vector_cosine_ops);
'''
with psycopg2.connect(**PG) as conn, conn.cursor() as cur:
    cur.execute(DDL)
print('schema ready')

In [ ]:
# ── Observability ingest: firing alerts + warning events ──
def thanos_alerts():
    r = requests.get(f'{THANOS}/api/v1/alerts',
                     headers={'Authorization': f'Bearer {SA_TOKEN}'}, verify=CA)
    r.raise_for_status()
    return r.json()['data']['alerts']

from kubernetes import client, config
config.load_incluster_config()
v1 = client.CoreV1Api()

def warning_events(namespaces=('aiops-demo-app', 'aap', 'aiops-events')):
    out = []
    for ns in namespaces:
        for ev in v1.list_namespaced_event(ns).items:
            if ev.type == 'Warning':
                out.append({'namespace': ns, 'reason': ev.reason,
                            'object': f'{ev.involved_object.kind}/{ev.involved_object.name}',
                            'message': ev.message})
    return out

alerts, events = thanos_alerts(), warning_events()
print(f'{len(alerts)} alerts, {len(events)} warning events')

In [ ]:
# ── Embed + store ──
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer('all-MiniLM-L6-v2')  # CPU-friendly, 384-dim

def incident_text(source, payload):
    return f"{source}: " + json.dumps(payload, default=str)[:2000]

def store(source, kind, namespace, subject, payload, rca=None, remediation=None, outcome=None):
    emb = embedder.encode(incident_text(source, payload)).tolist()
    with psycopg2.connect(**PG) as conn, conn.cursor() as cur:
        cur.execute(
            'INSERT INTO incidents (source, kind, namespace, subject, raw, rca, remediation, outcome, embedding)'
            ' VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s) RETURNING id',
            (source, kind, namespace, subject, json.dumps(payload, default=str),
             rca, remediation, outcome, str(emb)))
        return cur.fetchone()[0]

for a in alerts:
    store('alertmanager', a['labels'].get('alertname', 'unknown'),
          a['labels'].get('namespace'), a['labels'].get('pod') or a['labels'].get('node'), a)
for e in events:
    store('pod-event', e['reason'], e['namespace'], e['object'], e)
print('ingested')

In [ ]:
# ── Precedent retrieval: 'have we seen this before, and what fixed it?' ──
def similar_incidents(payload, k=3):
    emb = embedder.encode(incident_text('query', payload)).tolist()
    with psycopg2.connect(**PG) as conn, conn.cursor() as cur:
        cur.execute(
            'SELECT id, occurred_at, kind, namespace, subject, rca, remediation, outcome,'
            '       1 - (embedding <=> %s::vector) AS similarity'
            ' FROM incidents WHERE remediation IS NOT NULL'
            ' ORDER BY embedding <=> %s::vector LIMIT %s',
            (str(emb), str(emb), k))
        return cur.fetchall()

if alerts:
    for row in similar_incidents(alerts[0]):
        print(row)

In [ ]:
# ── NL incident summary — via Portkey ONLY (lesson L4) ──
def summarize(payload, precedents):
    prompt = (
        'You are an SRE assistant. Draft a concise incident summary:\n'
        f'CURRENT EVENT:\n{json.dumps(payload, default=str)[:3000]}\n\n'
        f'SIMILAR PAST INCIDENTS (with outcomes):\n{json.dumps(precedents, default=str)[:3000]}\n\n'
        'Cover: what happened, probable root cause, precedent-based '
        'recommendation, and a one-line suggested remediation playbook description.')
    r = requests.post(f'{PORTKEY}/v1/chat/completions',
                      json={'model': MODEL, 'max_tokens': 500,
                            'messages': [{'role': 'user', 'content': prompt}]})
    r.raise_for_status()
    return r.json()['choices'][0]['message']['content']

if alerts:
    print(summarize(alerts[0], [list(map(str, r)) for r in similar_incidents(alerts[0])]))

In [ ]:
# ── AAP inventory source export ──
# Writes precedent facts as an inventory the AAP 'AIOps Context' inventory
# source consumes (SCM source pointing at this path via the workbench PVC /
# the lightspeed-playbooks repo). Lightspeed generation prompts then carry
# hostvars like aiops_precedents — real context, not toy input.
def export_inventory(path='/opt/app-root/src/aap-inventory/incidents.json'):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with psycopg2.connect(**PG) as conn, conn.cursor() as cur:
        cur.execute(
            "SELECT kind, namespace, subject, rca, remediation, outcome"
            " FROM incidents WHERE outcome IS NOT NULL ORDER BY occurred_at DESC LIMIT 100")
        rows = cur.fetchall()
    hosts = {}
    for kind, ns, subject, rca, remediation, outcome in rows:
        key = f'{kind}.{ns or "cluster"}'
        hosts.setdefault(key, {'aiops_precedents': []})['aiops_precedents'].append(
            {'subject': subject, 'rca': rca, 'remediation': remediation, 'outcome': outcome})
    inv = {'_meta': {'hostvars': hosts}, 'aiops_context': {'hosts': list(hosts)}}
    with open(path, 'w') as f:
        json.dump(inv, f, indent=2, default=str)
    print(f'wrote {len(hosts)} context hosts → {path}')

export_inventory()

## Closing the loop

AAP's `embed-outcome` playbook posts each finished remediation (context, RCA, generated playbook, outcome) back into the `incidents` table — see `playbooks/examples/embed-outcome-pgvector.yml`. Re-run the ingest + export cells (or schedule them as an Elyra pipeline) and the next similar failure retrieves this precedent through `similar_incidents()`.